# Módulo 5 y Fase 0 Transversal: Gobierno de Datos (Habeas Data) y Viabilidad Actuarial de Decisiones de Vida

**Responsable OED:** Nelson Angel — *Ingeniero Industrial y Actuario | Líder de Gobierno de Datos y Datos Personales*  
**Rama del Proyecto:** `nean`  
**Módulos Conectados:** `src/data_governance.py`, `src/mod5_decisiones_vida.py` y `src/finance.py`

---

## Objetivo del Notebook
Este notebook interactivo cumple un doble propósito dentro de la arquitectura **100% Local (*Privacy by Design*)** del **Asesor Financiero e Inversiones OED**:

1. **Capa Transversal de Gobierno de Datos (`src/data_governance.py`):**
   * Auditar y demostrar el registro de **Consentimiento Informado Local** conforme a la **Ley 1581 de 2012** y el **Decreto 1377 de 2013**.
   * Ejecutar la **seudonimización irreversible (HMAC-SHA256)** de identificadores personales (PII), el **cifrado simétrico autenticado** en disco local (`data/processed/perfil_cifrado.oed`) y el ejercicio del **Derecho de Supresión / Olvido (Art. 8 Ley 1581/2012)** mediante sobrescritura criptográfica.
   * Modelar y visualizar la **Calculadora de Habeas Data Financiero (Ley 1266 de 2008 y Ley 2157 de 2021 - "Borra y Cuenta Nueva")** para determinar el tiempo máximo de permanencia de reportes negativos en centrales de riesgo (Datacrédito / TransUnion).
2. **Motor Actuarial y Estocástico de Decisiones Vitales (`src/mod5_decisiones_vida.py`):**
   * Proyectar y graficar en **Plotly** la **Curva Actuarial de Costos de Criar y Educar un Hijo en Colombia (0 a 23 años)** considerando inflación DANE ($5.2\%\text{ EA}$), inflación educativa ($7.5\%\text{ EA}$) y probabilidades actuariales de supervivencia (${}_tp_x$).
   * Ejecutar una **Simulación de Monte Carlo ($1.000$ escenarios estocásticos)** para comparar la viabilidad financiera y probabilidad de solvencia de tres decisiones estructurales: **Casarse / Unión Marital**, **Tener un Hijo** y **Tomar un Crédito Estructural (Vehículo TCO / Posgrado)**.

In [ ]:
# Configuración del entorno local y carga de módulos del proyecto OED
import sys
from pathlib import Path
import random
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Asegurar que la raíz del repositorio esté en sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.finance import cargar_configuracion, ea_a_em, cuota_fija_francesa
from src.data_governance import (
    TEXTO_AUTORIZACION_LEY_1581,
    seudonimizar_identificador,
    registrar_consentimiento_habeas_data,
    cifrar_payload_local,
    descifrar_payload_local,
    guardar_perfil_local_seguro,
    ejercer_derecho_olvido_supresion,
    calcular_permanencia_centrales_riesgo_ley_1266,
)
from src.mod5_decisiones_vida import evaluar_viabilidad_decision_vida

config = cargar_configuracion()
perfil_semilla = config["perfil_semilla_taller"]
macro = config["parametros_macro_colombia"]

print("✅ Módulos cargados correctamente desde:", ROOT_DIR)
print(f"📌 Año de vigencia regulatoria: {macro['anio_vigencia']} | IPC DANE: {macro['ipc_anual_dane']*100:.1f}% | Inflación Educativa: {macro['inflacion_educativa_anual']*100:.1f}%")

---
## Parte 1: Gobierno de Datos Personales y Financieros en Colombia (`src/data_governance.py`)

### 1.1 Consentimiento Informado, Seudonimización y Cifrado Local (Ley 1581 de 2012)
Conforme a la **Ley Estatutaria 1581 de 2012** y el **Decreto 1377 de 2013**, ningún dato financiero del usuario se procesa ni almacena en disco sin autorización previa, expresa e informada. Además, el identificador directo (cédula o correo) es reemplazado por un hash HMAC-SHA256 irreversible antes de persistir el archivo cifrado `.oed`.

In [ ]:
# 1. Registro auditable del Consentimiento Informado (Ley 1581 de 2012)
cedula_simulacion = "CC_1020304050_NELSON_DEMO"
consentimiento = registrar_consentimiento_habeas_data(
    usuario_id=cedula_simulacion,
    acepta_tratamiento=True,
    finalidad="Simulación actuarial y financiera 100% local - Taller OED",
)

df_consentimiento = pd.DataFrame([
    {"Campo de Auditoría Legal": "Consentimiento Válido", "Valor Registrado": str(consentimiento["consentimiento_valido"])},
    {"Campo de Auditoría Legal": "Identificador Original (PII)", "Valor Registrado": "[PROTEGIDO EN MEMORIA - NO SE GUARDA EN DISCO]"},
    {"Campo de Auditoría Legal": "ID Seudonimizado (HMAC-SHA256)", "Valor Registrado": consentimiento["usuario_seudonimizado"]},
    {"Campo de Auditoría Legal": "Marco Legal Aplicado", "Valor Registrado": consentimiento["marco_legal"]},
    {"Campo de Auditoría Legal": "Fecha y Hora UTC", "Valor Registrado": consentimiento["fecha_hora_utc"]},
    {"Campo de Auditoría Legal": "Firma de Integridad SHA-256", "Valor Registrado": consentimiento["hash_evidencia_sha256"][:32] + "..."},
])
df_consentimiento

In [ ]:
# 2. Demostración de Almacenamiento Cifrado Local, Descifrado Verificado y Derecho al Olvido (Art. 8 Ley 1581)
ruta_demo_cifrada = ROOT_DIR / "data" / "processed" / "perfil_demo_nelson.oed"

# Guardado seguro con consentimiento activo
resultado_guardado = guardar_perfil_local_seguro(
    perfil_datos=perfil_semilla,
    consentimiento=consentimiento,
    ruta_destino=ruta_demo_cifrada,
)

token_en_disco = ruta_demo_cifrada.read_text(encoding="utf-8")
payload_recuperado = descifrar_payload_local(token_en_disco)

print("🔒 Archivo cifrado creado en:", resultado_guardado["ruta_archivo"])
print("🔐 Muestra del token cifrado en disco (primeros 90 caracteres):", token_en_disco[:90] + "...")
print("✅ Usuario recuperado tras verificar firma HMAC:", payload_recuperado["perfil"]["usuario_id"])

# Ejercicio del Derecho de Supresión / Olvido (sobrescritura con bytes aleatorios + borrado)
resultado_olvido = ejercer_derecho_olvido_supresion(ruta_archivo=ruta_demo_cifrada)
print("🗑️ Derecho al Olvido ejecutado:", resultado_olvido["mensaje"])
print("📁 ¿Existe el archivo en disco después de la supresión?:", ruta_demo_cifrada.exists())

### 1.2 Calculadora de Habeas Data Financiero: Permanencia en Centrales de Riesgo (Ley 1266/2008 y Ley 2157/2021)
Bajo la **Ley 1266 de 2008** y la **Ley 2157 de 2021**, el tiempo máximo que un reporte negativo puede permanecer en Datacrédito o TransUnion después de pagada la obligación es:
$$\text{Permanencia Post-Pago (meses)} = \min(2 \times \text{Meses en Mora},\; 48\text{ meses})$$
Adicionalmente, toda obligación insoluta (no pagada) caduca de pleno derecho al cumplir **8 años ($96$ meses)** desde el inicio de la mora.

In [ ]:
# Simulación paramétrica de tiempos de permanencia en Datacrédito / TransUnion (0 a 36 meses de mora)
registros_habeas = []
for mora in range(0, 37, 2):
    res = calcular_permanencia_centrales_riesgo_ley_1266(meses_en_mora=mora, obligacion_pagada=True)
    registros_habeas.append({
        "Meses en Mora": mora,
        "Meses de Permanencia del Reporte Negativo (Post-Pago)": res["meses_permanencia_post_pago"],
        "Regla Legal": "Doble del tiempo en mora (< 24m)" if mora < 24 else "Tope Legal Máximo (48 meses / 4 años)",
    })

df_habeas = pd.DataFrame(registros_habeas)
df_habeas.iloc[::3]

In [ ]:
# Visualización Plotly: Curva Legal de Permanencia del Reporte Negativo (Ley 1266/2008 y Ley 2157/2021)
fig_habeas = go.Figure()

fig_habeas.add_trace(go.Scatter(
    x=df_habeas["Meses en Mora"],
    y=df_habeas["Meses de Permanencia del Reporte Negativo (Post-Pago)"],
    mode="lines+markers",
    name="Permanencia Legal Post-Pago (min(2x, 48m))",
    line=dict(color="#1E88E5", width=3),
    marker=dict(size=7),
))

fig_habeas.add_hline(
    y=48,
    line_dash="dash",
    line_color="#D32F2F",
    annotation_text="Tope Máximo Post-Pago: 48 meses (4 años - Ley 1266/2008)",
    annotation_position="bottom right",
)

fig_habeas.add_vline(
    x=24,
    line_dash="dot",
    line_color="#F57C00",
    annotation_text="Umbral de 24 meses de mora",
    annotation_position="top left",
)

fig_habeas.update_layout(
    title="Habeas Data Financiero en Colombia: Permanencia del Reporte Negativo tras el Pago (Ley 1266/2008 & Ley 2157/2021)",
    xaxis_title="Tiempo en Mora antes del Pago (Meses)",
    yaxis_title="Permanencia Máxima del Reporte Negativo (Meses)",
    template="plotly_white",
    height=430,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_habeas.show()

---
## Parte 2: Curva Actuarial de Costos de Criar y Educar un Hijo en Colombia (0 a 23 Años)

El motor actuarial de `src/mod5_decisiones_vida.py` divide el ciclo de crianza y formación en tres etapas sujetas a inflaciones diferenciales en Colombia y ponderadas por la probabilidad actuarial de supervivencia/continuidad ${}_tp_x$:

1. **Primera Infancia y Gestación (0 a 4 años):** Indexada al IPC general DANE ($5.2\%\text{ EA}$).
2. **Jardín, Primaria y Bachillerato (5 a 17 años):** Indexada a la inflación educativa histórica en Colombia ($7.5\%\text{ EA} \approx \text{IPC} + 2.3\%$). 
3. **Educación Superior / Universidad (18 a 22 años):** Indexada a la inflación educativa ($7.5\%\text{ EA}$).

$$VPA = \sum_{t=0}^{22} \frac{E[\text{Costo}_t] \cdot (1 + \pi_{\text{sector}})^t}{(1 + r_{\text{descuento}})^t} \cdot {}_tp_x$$

In [ ]:
# Construcción detallada año a año (0 a 22 años cumplidos -> 23 años de horizonte) para los 3 niveles de costo
ipc = float(macro["ipc_anual_dane"])
inf_edu = float(macro["inflacion_educativa_anual"])
tasa_descuento = 0.095
escalas = {"Bajo": 0.65, "Medio": 1.0, "Alto": 1.85}

filas_curva_hijo = []
for nombre_nivel, factor_escala in escalas.items():
    prob_supervivencia = 1.0
    vpa_acumulado = 0.0
    for edad in range(0, 23):
        prob_supervivencia *= 0.9985
        if edad <= 4:
            etapa = "1. Primera Infancia (0-4 años)"
            costo_mensual_hoy = 1_150_000.0 * factor_escala
            inf_aplicada = ipc
        elif edad <= 17:
            etapa = "2. Colegio (5-17 años)"
            costo_mensual_hoy = 1_650_000.0 * factor_escala
            inf_aplicada = inf_edu
        else:
            etapa = "3. Universidad (18-22 años)"
            costo_mensual_hoy = 2_450_000.0 * factor_escala
            inf_aplicada = inf_edu

        flujo_anual_hoy = costo_mensual_hoy * 12.0
        flujo_nominal_futuro = flujo_anual_hoy * ((1.0 + inf_aplicada) ** edad)
        vp_actuarial_anual = (flujo_nominal_futuro / ((1.0 + tasa_descuento) ** edad)) * prob_supervivencia
        vpa_acumulado += vp_actuarial_anual

        filas_curva_hijo.append({
            "Nivel de Gasto": nombre_nivel,
            "Edad (Años)": edad,
            "Etapa Formativa": etapa,
            "Flujo Anual Pesos de Hoy (COP)": flujo_anual_hoy,
            "Flujo Nominal Proyectado con Inflación (COP)": flujo_nominal_futuro,
            "Valor Presente Actuarial Anual (COP)": vp_actuarial_anual,
            "VPA Acumulado (COP)": vpa_acumulado,
        })

df_curva_hijo = pd.DataFrame(filas_curva_hijo)
resumen_niveles = df_curva_hijo.groupby("Nivel de Gasto", as_index=False).agg(
    Costo_Nominal_Total_23a=("Flujo Nominal Proyectado con Inflación (COP)", "sum"),
    VPA_Total_Hoy_COP=("Valor Presente Actuarial Anual (COP)", "sum"),
)
resumen_niveles["Costo Nominal Total 23a (COP)"] = resumen_niveles["Costo_Nominal_Total_23a"].map(lambda x: f"${x:,.0f}")
resumen_niveles["VPA Total Hoy (COP)"] = resumen_niveles["VPA_Total_Hoy_COP"].map(lambda x: f"${x:,.0f}")
resumen_niveles[["Nivel de Gasto", "Costo Nominal Total 23a (COP)", "VPA Total Hoy (COP)"]]

In [ ]:
# Gráfico Interactivo Plotly: Curva Actuarial de Costos (0 a 23 años) Nominal vs. Valor Presente Actuarial
fig_hijo = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Flujo Anual Proyectado con Inflación Educ./IPC (Escenario Medio)",
        "Valor Presente Actuarial Acumulado (0 a 23 años por Nivel de Costo)",
    ),
)

df_medio = df_curva_hijo[df_curva_hijo["Nivel de Gasto"] == "Medio"]
colores_etapa = {
    "1. Primera Infancia (0-4 años)": "#26A69A",
    "2. Colegio (5-17 años)": "#42A5F5",
    "3. Universidad (18-22 años)": "#7E57C2",
}

for etapa, df_et in df_medio.groupby("Etapa Formativa"):
    fig_hijo.add_trace(
        go.Bar(
            x=df_et["Edad (Años)"],
            y=df_et["Flujo Nominal Proyectado con Inflación (COP)"] / 1e6,
            name=etapa,
            marker_color=colores_etapa[etapa],
        ),
        row=1,
        col=1,
    )

fig_hijo.add_trace(
    go.Scatter(
        x=df_medio["Edad (Años)"],
        y=df_medio["Valor Presente Actuarial Anual (COP)"] / 1e6,
        mode="lines+markers",
        name="VP Actuarial Anual (Descontado al 9.5% EA)",
        line=dict(color="#D32F2F", width=2.5, dash="dot"),
    ),
    row=1,
    col=1,
)

colores_nivel = {"Bajo": "#66BB6A", "Medio": "#1E88E5", "Alto": "#E53935"}
for nivel in ["Bajo", "Medio", "Alto"]:
    df_niv = df_curva_hijo[df_curva_hijo["Nivel de Gasto"] == nivel]
    fig_hijo.add_trace(
        go.Scatter(
            x=df_niv["Edad (Años)"],
            y=df_niv["VPA Acumulado (COP)"] / 1e6,
            mode="lines+markers",
            name=f"VPA Acumulado - Nivel {nivel}",
            line=dict(color=colores_nivel[nivel], width=3),
        ),
        row=1,
        col=2,
    )

fig_hijo.update_xaxes(title_text="Edad del Hijo (Años)", row=1, col=1)
fig_hijo.update_xaxes(title_text="Edad del Hijo (Años)", row=1, col=2)
fig_hijo.update_yaxes(title_text="Millones de Pesos (COP)", row=1, col=1)
fig_hijo.update_yaxes(title_text="VPA Acumulado (Millones COP)", row=1, col=2)
fig_hijo.update_layout(
    title="Modelo Actuarial OED: Costo de Criar y Educar un Hijo en Colombia (0 a 23 años)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=-0.28, xanchor="center", x=0.5),
)
fig_hijo.show()

---
## Parte 3: Simulación de Monte Carlo ($1.000$ Escenarios) para Casarse, Tener Hijos y Tomar un Crédito

Ahora integramos el estado financiero del hogar (proveniente de los módulos anteriores del taller: `ingreso_neto_hogar_mensual_cop = 9_115_000` y `excedente_mensual_actual_cop = 2_145_000`) y evaluamos con `evaluar_viabilidad_decision_vida` de `src/mod5_decisiones_vida.py` las tres grandes decisiones vitales sometidas a **$1.000$ choques estocásticos simultáneos** de ingresos ($\sigma_{\text{ingreso}} = 8\%$) e inflación de gastos ($\sigma_{\text{gasto}} = 7\%$):

1. **`CASARSE`**: Evalúa el costo de boda ($\$18.000.000\text{ COP}$) frente al ahorro mensual por economías de escala de compartir gastos de vivienda y servicios con la pareja, más la recomendación legal sobre **Sociedad Conyugal vs. Capitulaciones Matrimoniales**.
2. **`HIJOS`**: Evalúa el impacto mensual inmediato de primera infancia, el fondo educativo mensual para cubrir el $100\%$ de la universidad a los 18 años y la póliza de **Seguro de Vida Temporal** por el $VPA$ total.
3. **`CREDITO`**: Evalúa un crédito estructural de $\$45.000.000\text{ COP}$ al $16.5\%\text{ EA}$ a 48 meses sumando el **Costo Total de Propiedad (*TCO*)** oculto mensual (SOAT, impuestos, seguro todo riesgo y mantenimiento) y el derecho a endosar póliza de vida deudor (**Ley 1328 de 2009**).

In [ ]:
# Parámetros del Caso Maestro OED (Ingreso Neto Hogar: $9.115.000 COP | Excedente Mensual: $2.145.000 COP)
ingreso_neto_mensual_cop = 9_115_000.0
excedente_mensual_actual_cop = 2_145_000.0
liquidez_emergencia_cop = float(perfil_semilla["liquidez_actual_fondo_emergencia_cop"])

decisiones = ["CASARSE", "HIJOS", "CREDITO"]
resultados_mod5 = {}

for dec in decisiones:
    resultados_mod5[dec] = evaluar_viabilidad_decision_vida(
        tipo_decision=dec,
        ingreso_neto_hogar_mensual_cop=ingreso_neto_mensual_cop,
        excedente_mensual_actual_cop=excedente_mensual_actual_cop,
        liquidez_emergencia_cop=liquidez_emergencia_cop,
        nivel_costo_hijo="Medio",
        ingreso_pareja_mensual_cop=4_500_000.0,
        costo_boda_cop=18_000_000.0,
        monto_credito_estructural_cop=45_000_000.0,
        tasa_ea_credito=0.165,
        plazo_credito_meses=48,
        iteraciones_montecarlo=1000,
        config=config,
    )

df_resumen_decisiones = pd.DataFrame(list(resultados_mod5.values()))
df_resumen_decisiones

In [ ]:
# Generación de las 1.000 trayectorias de Monte Carlo para visualizar la distribución completa en Plotly
registros_mc = []
etiquetas_legibles = {
    "CASARSE": "1. Casarse / Unión Marital (Con Sinergia de Hogar)",
    "HIJOS": "2. Tener un Hijo (0 a 23 años - Nivel Medio)",
    "CREDITO": "3. Crédito Estructural Vehículo/Posgrado ($45M + TCO)",
}

for dec in decisiones:
    impacto = resultados_mod5[dec]["impacto_flujo_mensual_inmediato_cop"]
    rng = random.Random(20261110)
    for i in range(1000):
        choque_ingreso = rng.gauss(1.0, 0.08)
        choque_gasto = rng.gauss(1.0, 0.07)
        flujo_neto_simulado = (
            excedente_mensual_actual_cop * choque_ingreso
            - impacto * choque_gasto
            + (liquidez_emergencia_cop / 24.0)
        )
        registros_mc.append({
            "Decisión Vital": etiquetas_legibles[dec],
            "Código": dec,
            "Iteración": i + 1,
            "Flujo de Caja Neto Mensual Post-Decisión (COP)": flujo_neto_simulado,
            "Estado Escenario": "Solvente (Flujo >= $0)" if flujo_neto_simulado >= 0 else "Déficit (Flujo < $0)",
        })

df_mc = pd.DataFrame(registros_mc)

fig_mc = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.62, 0.38],
    subplot_titles=(
        "Distribución de Flujo de Caja Mensual Simulado (1.000 Escenarios Monte Carlo)",
        "Probabilidad de Éxito / Solvencia (%)",
    ),
)

colores_dec = {
    "CASARSE": "#2E7D32",
    "HIJOS": "#1565C0",
    "CREDITO": "#EF6C00",
}

for dec in decisiones:
    sub = df_mc[df_mc["Código"] == dec]
    fig_mc.add_trace(
        go.Histogram(
            x=sub["Flujo de Caja Neto Mensual Post-Decisión (COP)"] / 1e6,
            name=etiquetas_legibles[dec],
            marker_color=colores_dec[dec],
            opacity=0.68,
            nbinsx=40,
        ),
        row=1,
        col=1,
    )

# Línea de solvencia crítica (Flujo Neto = $0 COP)
fig_mc.add_vline(
    x=0.0,
    line_dash="dash",
    line_color="#B71C1C",
    line_width=2.5,
    annotation_text="Umbral de Insolvencia ($0 COP)",
    annotation_position="top left",
    row=1,
    col=1,
)

# Gráfico de barras de Probabilidad de Éxito Monte Carlo
fig_mc.add_trace(
    go.Bar(
        x=[etiquetas_legibles[d].split(" (")[0] for d in decisiones],
        y=[resultados_mod5[d]["probabilidad_exito_montecarlo_pct"] for d in decisiones],
        text=[f"{resultados_mod5[d]['probabilidad_exito_montecarlo_pct']:.1f}%<br>({resultados_mod5[d]['dictamen_viabilidad']})" for d in decisiones],
        textposition="auto",
        marker_color=[colores_dec[d] for d in decisiones],
        name="Probabilidad de Solvencia (%)",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig_mc.add_hline(
    y=80.0,
    line_dash="dot",
    line_color="#2E7D32",
    annotation_text="Meta Solvente (80%)",
    row=1,
    col=2,
)

fig_mc.update_layout(
    barmode="overlay",
    title="Simulación Estocástica de Monte Carlo (1.000 Escenarios): Solvencia ante Decisiones de Vida",
    template="plotly_white",
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=-0.26, xanchor="center", x=0.5),
)
fig_mc.update_xaxes(title_text="Flujo de Caja Mensual Libre Post-Decisión (Millones COP)", row=1, col=1)
fig_mc.update_yaxes(title_text="Frecuencia (Número de Escenarios)", row=1, col=1)
fig_mc.update_yaxes(title_text="Probabilidad de Éxito (%)", range=[0, 110], row=1, col=2)
fig_mc.show()

### 3.1 Evaluación de Sensibilidad: ¿Qué pasa si el Hogar Toma las Decisiones con un Excedente Ajustado?
Para apoyar la toma de decisiones del usuario, simulamos cómo cambia la **Probabilidad de Solvencia de Monte Carlo** cuando el excedente mensual disponible varía desde $\$600.000\text{ COP}$ hasta $\$2.400.000\text{ COP}$.

In [ ]:
# Curva de sensibilidad de la probabilidad de éxito según el excedente mensual disponible
excedentes_prueba = list(range(600_000, 2_500_000, 150_000))
filas_sensibilidad = []

for exc in excedentes_prueba:
    for dec in decisiones:
        res_sens = evaluar_viabilidad_decision_vida(
            tipo_decision=dec,
            ingreso_neto_hogar_mensual_cop=ingreso_neto_mensual_cop,
            excedente_mensual_actual_cop=float(exc),
            liquidez_emergencia_cop=liquidez_emergencia_cop,
            nivel_costo_hijo="Medio",
            ingreso_pareja_mensual_cop=4_500_000.0,
            costo_boda_cop=18_000_000.0,
            monto_credito_estructural_cop=45_000_000.0,
            tasa_ea_credito=0.165,
            plazo_credito_meses=48,
            iteraciones_montecarlo=1000,
            config=config,
        )
        filas_sensibilidad.append({
            "Excedente Mensual Actual (Millones COP)": exc / 1e6,
            "Decisión Vital": etiquetas_legibles[dec],
            "Probabilidad de Solvencia (%)": res_sens["probabilidad_exito_montecarlo_pct"],
        })

df_sens = pd.DataFrame(filas_sensibilidad)
fig_sens = px.line(
    df_sens,
    x="Excedente Mensual Actual (Millones COP)",
    y="Probabilidad de Solvencia (%)",
    color="Decisión Vital",
    markers=True,
    color_discrete_sequence=["#2E7D32", "#1565C0", "#EF6C00"],
    title="Curva de Fragilidad Actuarial: Probabilidad de Éxito vs. Excedente Mensual Disponible",
)
fig_sens.add_hline(y=80.0, line_dash="dash", line_color="#2E7D32", annotation_text="Zona Viable Solvente (>= 80%)")
fig_sens.add_hline(y=55.0, line_dash="dot", line_color="#D32F2F", annotation_text="Umbral de Riesgo Crítico (< 55%)")
fig_sens.update_layout(template="plotly_white", height=450)
fig_sens.show()

### Q&A
* **¿Cómo garantiza el sistema el cumplimiento de la Ley 1581 de 2012 y la Ley 1266 de 2008 / 2157 de 2021?**
  El módulo transversal `src/data_governance.py` bloquea cualquier escritura en disco si `consentimiento_valido` es `False`, seudonimiza el identificador del titular mediante `HMAC-SHA256`, cifra el perfil financiero localmente (`OED-HMAC-SHA256-LOCAL`) y permite ejercer el Derecho de Supresión sobrescribiendo el archivo con bytes aleatorios antes de eliminarlo. Para reportes negativos en centrales de riesgo, aplica exactamente la regla $\min(2 \times \text{meses en mora}, 48\text{ meses})$ post-pago y la caducidad a los $96$ meses (8 años) para obligaciones insolutas.
* **¿Cuál es el Valor Presente Actuarial ($VPA$) de criar y educar un hijo de 0 a 23 años en Colombia y qué cobertura requiere?**
  En el escenario de costo **Medio**, el Valor Presente Actuarial ($VPA$) descontado al $9.5\%\text{ EA}$ con inflación educativa del $7.5\%\text{ EA}$ e IPC del $5.2\%\text{ EA}$ asciende a **$\$369.048.420\text{ COP}$** (equivalente a constituir una póliza de seguro de vida temporal por ese valor y un ahorro educativo nivelado de **$\$929.702\text{ COP}$ mensuales** durante 18 años para cubrir el $100\%$ de la universidad sin deuda).

### Data Analysis Key Findings
* **Asimetría de Costos por Nivel Socioeconómico/Educativo (0 a 23 años):** El Valor Presente Actuarial ($VPA$) de criar un hijo pasa de **$\$239.881.473\text{ COP}$** en el escenario Bajo ($0.65\times$) a **$\$369.048.420\text{ COP}$** en el escenario Medio ($1.0\times$) y alcanza **$\$682.739.577\text{ COP}$** en el escenario Alto ($1.85\times$), impulsado principalmente por la inflación educativa compuesta del $7.5\%\text{ EA}$ entre los 5 y 22 años.
* **Efecto de Economías de Escala al Casarse (`CASARSE`):** Con un ingreso neto del hogar de **$\$9.115.000\text{ COP}$** y un excedente mensual de **$\$2.145.000\text{ COP}$**, para una boda de **$\$18.000.000\text{ COP}$** amortizada en 24 meses ($\$750.000\text{ COP/mes}$), la sinergia de compartir gastos fijos del hogar absorbe completamente el costo mensualizado, logrando un **$100.0\%$ de probabilidad de solvencia** (`VIABLE_SOLVENTE`).
* **Impacto del Costo Total de Propiedad (*TCO*) en Créditos Estructurales (`CREDITO`):** Un crédito de **$\$45.000.000\text{ COP}$** al $16.5\%\text{ EA}$ a 48 meses más los costos ocultos de tenencia vehicular ($0.8\%$ mensual = **$\$360.000\text{ COP/mes}$** en SOAT, impuestos, seguro todo riesgo y mantenimiento) genera un impacto mensual de **$\$1.620.833\text{ COP/mes}$** ($VPA$ de **$\$77.799.989\text{ COP}$**), logrando un **$99.7\%$ de probabilidad de solvencia** con el excedente de $\$2.145.000\text{ COP}$.

### Insights or Next Steps
* **Blindaje Actuarial Previo a Decisiones Familiares:** Antes de asumir la decisión de `HIJOS` o `CREDITO`, el hogar debe automatizar el aporte mensual de **$\$929.702\text{ COP}$** en un FIC/Cuenta AFC para el fondo universitario y endosar su propia póliza de vida deudor (Ley 1328 de 2009) para reducir hasta un $40\%$ el costo de seguros bancarios.
* **Integración en la Demo Final OED (Etapa 5):** Conectar directamente la salida del Módulo 3 (flujo de caja liberado por Catalina tras el método Avalancha) y del Módulo 4 (cuota hipotecaria VIS validada por Jesi bajo la Ley 546 de 1999) como entrada de `excedente_mensual_actual_cop` en este notebook para dictaminar en vivo la viabilidad conjunta del Caso Maestro OED.